In [0]:
%sql
USE CATALOG shopsphere;

CREATE TABLE IF NOT EXISTS shopsphere.silver.payments(
   payment_id LONG,
    order_id LONG,
    payment_method STRING,
    payment_amount DECIMAL,
    payment_status STRING,
    payment_date TIMESTAMP,
    updated_at TIMESTAMP
);

CREATE TABLE IF NOT EXISTS shopsphere.quarantine.payments(
    payment_id LONG,
    order_id LONG,
    payment_method STRING,
    payment_amount DECIMAL,
    payment_status STRING,
    payment_date TIMESTAMP,
    updated_at TIMESTAMP
)
USING DELTA;

In [0]:
from pyspark.sql.functions import *
from delta.tables import *

df = spark.read.table("shopsphere.bronze.payments")


In [0]:
#remove duplicates
df = df.dropDuplicates()

#validate payment amount
valid_payment = df.filter(
                    (col('payment_amount') > 0)
                    & (col('payment_amount').isNotNull())
                    & (round(col('payment_amount'), 2) == col('payment_amount'))
)

df_quarantine = df.filter(
                    (~((col('payment_amount') > 0)
                    & (col('payment_amount').isNotNull())
                    & (round(col('payment_amount'), 2) == col('payment_amount'))))).withColumn("validation_status", lit("invalid_payment"))


In [0]:
#quarantine rejected rows from validation
quarantine_table = DeltaTable.forName(spark,"shopsphere.quarantine.payments")

quarantine_table.alias("target").merge(df_quarantine.alias("source"),
                                       "target.payment_id = source.payment_id").\
                                        whenNotMatchedInsertAll().\
                                        withSchemaEvolution().\
                                        execute()

In [0]:
#Standardize payment method
standard_payment = valid_payment.withColumn("payment_method", initcap(trim(col("payment_method"))))

#Standardize payment status
standard_payment = standard_payment.withColumn("payment_status", initcap(trim(col("payment_status"))))

In [0]:
#write transformed data to silver table
silver_table = DeltaTable.forName(spark,"shopsphere.silver.payments")

silver_table.alias("target").merge(standard_payment.alias("source"), \
                                       "target.payment_id = source.payment_id" ).\
                        whenNotMatchedInsertAll().\
                        whenMatchedUpdateAll().\
                        execute()

In [0]:
%sql

SELECT * FROM shopsphere.silver.payments
WHERE payment_status in ("Failed","Success","Refunded")